In [33]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv() 

True

In [34]:
#Document Loading
loader = PyPDFLoader("telecom_guide.pdf")
pages = loader.load()

In [35]:
print(pages[5].page_content)

Telecom Technical Reference Guide  - Internal Use Only
5. SIM Card Technology
The Subscriber Identity Module (SIM) is the secure element that identifies a customer on the network. It stores the
International Mobile Subscriber Identity (IMSI), authentication keys, and basic service configuration.
Form Factors: SIM cards have evolved from the original credit-card-sized (1FF) format through mini-SIM (2FF),
micro-SIM (3FF), and nano-SIM (4FF). Today the vast majority of smartphones use nano-SIM. Some industrial
IoT devices still use the larger 2FF format for robustness.
eSIM: The embedded SIM (eSIM) is a chip soldered directly into the device during manufacture. Unlike a physical
SIM it can be reprogrammed over the air. The customer receives a QR code or activation code that loads a profile
onto the eSIM. eSIM supports multiple profiles simultaneously, allowing the user to switch between carriers or
maintain a local SIM while roaming without physically swapping cards. Most flagship smartph

In [36]:
# Text Splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
    )

texts = text_splitter.split_documents(pages)
print(texts)


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'), Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.'), D

In [37]:
# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
print(embeddings)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5840.31it/s]


model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} query_encode_kwargs={} multi_process=False show_progress=False


In [38]:
# VectorDB
vectorDB = Chroma.from_documents(texts, embeddings)
print(vectorDB)

In [39]:
# Retrieval
retreiver = vectorDB.as_retriever(search_kwargs={"k": 3})

test = "what is volte and how it improves call quality?"
retreived = retreiver.invoke(test)
print(retreived)

[Document(id='87d5348a-5413-4cb7-8c9f-3411ef30c883', metadata={'producer': 'PyPDF', 'page': 6, 'total_pages': 9, 'source': 'telecom_guide.pdf', 'creationdate': '2026-04-30T14:16:24+00:00', 'creator': 'PyPDF', 'page_label': '7'}, page_content='legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and\nvoice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an\naccount that has VoLTE activated.\nEnabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On\niPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent'), Document(id='ed9b636d-0e1b-4a28-937a-9f5f26f970dc', metadata={'creationdate': '2026-04-30T14:16:24+00:00', 'total_pages': 9, 'source': 'telecom_guide.pdf', 'producer': 'PyPDF', 'creator': 'PyPDF', 'page_label': '7', 'page': 6}, page_content='legacy calls), faster call setup 

In [ ]:
# Prompt

system_prompt = """
You are a helpful assistant that answers questions about company's guide. Reply from available data if not clearly say so.

Context:
{context}
"""

user_prompt = "{question}"

In [44]:
# LLM call
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

llm = GoogleGenerativeAI(
    model_name="gemma-4-31b",
    temperature=0.1,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt),
])

chain = (
    {"context": retreiver | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

question = "what is volte and how it improves call quality?"
answer = chain.invoke(question)
print(answer)



Unexpected argument 'model_name' provided to GoogleGenerativeAI. Did you mean: 'model'?


ValidationError: 1 validation error for GoogleGenerativeAI
model
  Field required [type=missing, input_value={'model_name': 'gemma-4-3...SecretStr('**********')}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing